In [1]:
!pip install couchbase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 5.3 MB/s  0:00:00m 3.6 MB/s eta 0:00:01m


In [4]:
# docker run -d --name couchbase-server \
#  -p 8091-8094:8091-8094 \
#  -p 11210:11210 \
#  couchbase/server:latest
#
# http://localhost:8091

from datetime import timedelta
from couchbase.auth import PasswordAuthenticator
from couchbase.cluster import Cluster
from couchbase.options import ClusterOptions, QueryOptions

# 1. Connection Details
endpoint = "couchbase://localhost"
username = "Administrator"
password = "password"  # Use the password you set during setup
bucket_name = "test-bucket"

# 2. Connect to Cluster
auth = PasswordAuthenticator(username, password)
cluster = Cluster(endpoint, ClusterOptions(auth))

# Wait until the cluster is ready for use (essential for Docker)
cluster.wait_until_ready(timedelta(seconds=5))

# 3. Access Bucket and Collection
cb = cluster.bucket(bucket_name)
# Using the default collection for this demo
coll = cb.default_collection()

def run_demo():
    print("--- Starting Couchbase Python Demo ---")

    # CREATE (Upsert)
    user_id = "user:123"
    user_data = {
        "type": "user",
        "name": "Jane Doe",
        "email": "jane@example.com",
        "interests": ["python", "nosql", "docker"]
    }
    coll.upsert(user_id, user_data)
    print(f"Created/Updated document: {user_id}")

    # READ
    result = coll.get(user_id)
    print(f"Retrieved Name: {result.content_as[dict]['name']}")

    # UPDATE (Replace)
    user_data["interests"].append("couchbase")
    coll.replace(user_id, user_data)
    print("Updated interests to include 'couchbase'")

    # SQL++ QUERY (N1QL)
    # Requires an Index (run 'CREATE PRIMARY INDEX ON `test-bucket`' in Web UI first)
    try:
        query = f"SELECT name, email FROM `{bucket_name}` WHERE ANY i IN interests SATISFIES i = $interest END"
        rows = cluster.query(query, QueryOptions(positional_parameters=["couchbase"]))
        
        print("\nUsers interested in Couchbase:")
        for row in rows:
            print(f"- {row['name']} ({row['email']})")
    except Exception as e:
        print(f"\nQuery failed: {e} (Did you create a Primary Index?)")

    print("\n--- Demo Complete ---")

if __name__ == "__main__":
    run_demo()

--- Starting Couchbase Python Demo ---
Created/Updated document: user:123
Retrieved Name: Jane Doe
Updated interests to include 'couchbase'

Users interested in Couchbase:

Query failed: InternalServerFailureException(<ec=5, category=couchbase.common, message=internal_server_failure (5), context=QueryErrorContext({'last_dispatched_to': '[::1]:8093', 'last_dispatched_from': '[::1]:59216', 'retry_attempts': 0, 'client_context_id': '2b0a89-af40-c549-7830-2398690f485d94', 'method': 'POST', 'path': '/query/service', 'http_status': 200, 'http_body': '{\n"requestID": "bc559293-386d-4896-80c2-d61c8e6c49e8",\n"clientContextID": "2b0a89-af40-c549-7830-2398690f485d94",\n"signature": {"name":"json","email":"json"},\n"results": [\n],\n"errors": [{"code":5010,"column":78,"line":1,"msg":"Error evaluating filter - cause: No value for named parameter $interest (near line 1, column 78)."}],\n"status": "fatal"\n}\n', 'first_error_code': 5010, 'first_error_message': 'Error evaluating filter - cause: No va